<a href="https://colab.research.google.com/github/rezzz59/Sentimen-Analysis-Aplikasi-Grab/blob/main/grab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install google-play-scrapper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.1 MB/s eta 0:00:00


In [11]:
import sys
!{sys.executable} -m pip install google-play-scraper



In [12]:
import pandas as pd
from google_play_scraper import Sort, reviews

result, countinuation_token = reviews(
    'com.grabtaxi.passenger',
    lang = 'id',
    country = 'id',
    sort = Sort.NEWEST,
    count= 11000,
)

df = pd.DataFrame(result)

df = df[['content', 'score']]

df.to_csv('grab_review_raw.csv', index = False)

In [13]:
df

,content,score
0,"tampilan barunya merepotkan, sulit dipahami. p...",1
1,bagus pelayanannya,5
2,ramah dan baik',5
3,bagusss lebih murah dr pada gojek,5
4,kendaraan banyak gak mau ambil.. sngat d sayan...,1
...,...,...
10995,ramah & sopan santun,5
10996,Pelayanan nya cepat,5
10997,bagus,5
10998,terima kasih derever semoga selau amanah,5


In [14]:
import re
import string

slang_dict = {
    "ga": "tidak", "gak": "tidak", "gakk": "tidak", "nggak": "tidak",
    "yg": "yang", "dr": "dari", "bgt": "banget", "kl": "kalau",
    "udh": "sudah", "udah": "sudah", "aja": "saja", "jd": "jadi",
    "tp": "tapi", "pake": "pakai", "sdh": "sudah", "aja": "saja",
    "dapet": "dapat", "ilang": "hilang", "lemot": "lambat", "gercep": "cepat",
}

def clean_text(text):
  text = text.lower()

  #menghapus link, mention dan tag
  text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
  text = text = re.sub(r'@\w+|\#', '', text)

  #menghapus angka dan tanda bca
  text = text.translate(str.maketrans('', '', string.punctuation))
  text = re.sub(r'\d+', '', text)

  #menghapus emoji
  text = text.encode('ascii', 'ignore').decode('ascii')

  #normalisasi kata & tokenizing
  words = text.split()
  cleaned_words = [slang_dict.get(w, w) for w in words]

  return " ".join(cleaned_words)

df['content_cleaned'] = df['content'].apply(clean_text)
df = df[df['content_cleaned'] != '']


In [15]:
def labeling(score):
  if score > 3:
    return "positif"
  if score < 2:
    return 'negatif'
  else:
    return 'netral'

df['label'] = df['score'].apply(labeling)

/tmp/ipykernel_2795/1599349615.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['score'].apply(labeling)


In [16]:
df

,content,score,content_cleaned,label
0,"tampilan barunya merepotkan, sulit dipahami. p...",1,tampilan barunya merepotkan sulit dipahami pes...,negatif
1,bagus pelayanannya,5,bagus pelayanannya,positif
2,ramah dan baik',5,ramah dan baik,positif
3,bagusss lebih murah dr pada gojek,5,bagusss lebih murah dari pada gojek,positif
4,kendaraan banyak gak mau ambil.. sngat d sayan...,1,kendaraan banyak tidak mau ambil sngat d sayan...,negatif
...,...,...,...,...
10995,ramah & sopan santun,5,ramah sopan santun,positif
10996,Pelayanan nya cepat,5,pelayanan nya cepat,positif
10997,bagus,5,bagus,positif
10998,terima kasih derever semoga selau amanah,5,terima kasih derever semoga selau amanah,positif


In [17]:
df[['content_cleaned', 'score', 'label']]

,content_cleaned,score,label
0,tampilan barunya merepotkan sulit dipahami pes...,1,negatif
1,bagus pelayanannya,5,positif
2,ramah dan baik,5,positif
3,bagusss lebih murah dari pada gojek,5,positif
4,kendaraan banyak tidak mau ambil sngat d sayan...,1,negatif
...,...,...,...
10995,ramah sopan santun,5,positif
10996,pelayanan nya cepat,5,positif
10997,bagus,5,positif
10998,terima kasih derever semoga selau amanah,5,positif
